In [15]:
import numpy as np
import pandas as pd
import tensorflow as tf
import optuna
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

In [16]:
df = pd.read_csv("data/hgw_long_term.csv", parse_dates=["timestamp"])

df = df.sort_values(["gateway_id", "timestamp"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Gateways:", df["gateway_id"].nunique())

Shape: (438000, 31)
Gateways: 5


In [17]:
FEATURES = [
    "cpu_load", "mem_used_pct", "ping_latency", "packet_loss",
    "cwmp_rss_mb", "dhcp_rss_mb", "nemo_rss_mb",
    "wan_status", "cpu_slope_6h", "ram_slope_6h",
    "cpu_mean_24h", "ram_mean_24h",
    "cpu_std_24h", "ram_std_24h",
    "wan_instability_6h", "health_score"
]

LABEL = "incident_in_72h"

In [18]:
LOOKBACK = 21 * 24 * 2
SUBSAMPLE = 2 * 2
SEQ_LEN = LOOKBACK // SUBSAMPLE

def build_sequences(df):
    X, y = [], []

    for gw, group in df.groupby("gateway_id"):
        group = group.sort_values("timestamp")

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(group[FEATURES].fillna(0))

        labels = group[LABEL].values

        for i in range(LOOKBACK, len(group)):
            seq = X_scaled[i-LOOKBACK:i:SUBSAMPLE]
            X.append(seq)
            y.append(labels[i])

    return np.array(X, dtype=np.float32), np.array(y)

X, y = build_sequences(df)

print(X.shape, y.mean())

(432960, 252, 16) 0.12348253880266076


In [19]:
split = int(len(X) * 0.75)

X_tr, X_te = X[:split], X[split:]
y_tr, y_te = y[:split], y[split:]

print("Train positive:", y_tr.mean())
print("Test positive:", y_te.mean())

Train positive: 0.12368810051736881
Test positive: 0.12286585365853658


In [20]:
pos = np.where(y_tr == 1)[0]
neg = np.where(y_tr == 0)[0]

neg_keep = np.random.choice(neg, size=len(pos)*3, replace=False)
keep = np.concatenate([pos, neg_keep])

X_tr = X_tr[keep]
y_tr = y_tr[keep]

print("Balanced:", X_tr.shape)

Balanced: (160656, 252, 16)


In [21]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-6, 1-1e-6)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        return -tf.reduce_mean(alpha * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss

In [22]:
from tensorflow.keras.layers import Layer
import tensorflow.keras.backend as K

class AttentionLayer(Layer):
    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer="glorot_uniform")
        self.b = self.add_weight(shape=(input_shape[1], 1),
                                 initializer="zeros")

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)
        a = K.softmax(e, axis=1)
        return K.sum(x * a, axis=1)

In [23]:
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model

def build_model(units1, units2, dropout, lr):
    inp = Input(shape=(SEQ_LEN, len(FEATURES)))

    x = LSTM(units1, return_sequences=True, dropout=dropout)(inp)
    x = LSTM(units2, return_sequences=True, dropout=dropout)(x)

    context = AttentionLayer()(x)

    x = Dense(32, activation="relu")(context)
    x = Dropout(dropout)(x)

    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=focal_loss(),
        metrics=[tf.keras.metrics.AUC(curve="PR", name="prauc")]
    )

    return model

In [24]:
def objective(trial):#optuna optimization function
    units1 = trial.suggest_int("units1", 32, 128)
    units2 = trial.suggest_int("units2", 16, 64)
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)

    model = build_model(units1, units2, dropout, lr)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_te, y_te),  # ✅ correct
        epochs=10,
        batch_size=128,
        verbose=0
    )

    y_pred = model.predict(X_te).flatten()
    pr = average_precision_score(y_te, y_pred)

    return pr

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best params:", study.best_params)

[I 2026-05-02 11:17:01,534] A new study created in memory with name: no-name-ae2e5a34-451d-400d-aa29-f0ce386109f2


3383/3383 ━━━━━━━━━━━━━━━━━━━━ 204s 60ms/step


[I 2026-05-02 13:09:25,940] Trial 0 finished with value: 0.8210440908431379 and parameters: {'units1': 88, 'units2': 36, 'dropout': 0.24378362072496143, 'lr': 0.0006384420872046867}. Best is trial 0 with value: 0.8210440908431379.


3383/3383 ━━━━━━━━━━━━━━━━━━━━ 213s 63ms/step


[I 2026-05-02 15:20:38,782] Trial 1 finished with value: 0.823775724299162 and parameters: {'units1': 75, 'units2': 61, 'dropout': 0.38669310552999137, 'lr': 0.00045207755379248296}. Best is trial 1 with value: 0.823775724299162.


3383/3383 ━━━━━━━━━━━━━━━━━━━━ 245s 72ms/step


[I 2026-05-02 17:45:02,898] Trial 2 finished with value: 0.8395052524232224 and parameters: {'units1': 91, 'units2': 45, 'dropout': 0.21058216956081582, 'lr': 0.0002343867525007501}. Best is trial 2 with value: 0.8395052524232224.


3383/3383 ━━━━━━━━━━━━━━━━━━━━ 223s 66ms/step


[I 2026-05-02 19:25:19,692] Trial 3 finished with value: 0.8576335310602409 and parameters: {'units1': 43, 'units2': 64, 'dropout': 0.27036055181138674, 'lr': 0.00047251016747211006}. Best is trial 3 with value: 0.8576335310602409.
[W 2026-05-02 20:29:21,724] Trial 4 failed with parameters: {'units1': 41, 'units2': 57, 'dropout': 0.4780697491620271, 'lr': 0.0005304055388510389} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\bnajjar\AppData\Local\anaconda3\envs\hgw_project\lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\bnajjar\AppData\Local\Temp\ipykernel_12136\3227741694.py", line 9, in objective
    history = model.fit(
  File "c:\Users\bnajjar\AppData\Local\anaconda3\envs\hgw_project\lib\site-packages\keras\src\utils\traceback_utils.py", line 117, in error_handler
    return fn(*args, **kwargs)
  File "c:\Users\bnajjar\AppData\Local\anaconda3\envs\hgw_pr

KeyboardInterrupt: 

In [ ]:
params = study.best_params

model = build_model(
    params["units1"],
    params["units2"],
    params["dropout"],
    params["lr"]
)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_te, y_te),
    epochs=20,
    batch_size=128
)

Epoch 1/20
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 468s 429ms/step - auc: 0.8770 - loss: 0.0193 - val_auc: 0.0000e+00 - val_loss: 0.0089
Epoch 2/20
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 393s 369ms/step - auc: 0.9175 - loss: 0.0155 - val_auc: 0.0000e+00 - val_loss: 0.0068
Epoch 3/20
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 416s 390ms/step - auc: 0.9320 - loss: 0.0140 - val_auc: 0.0000e+00 - val_loss: 0.0080
Epoch 4/20
  59/1067 ━━━━━━━━━━━━━━━━━━━━ 5:16 314ms/step - auc: 0.9415 - loss: 0.0128

KeyboardInterrupt: 

In [ ]:
y_prob = model.predict(X_te).flatten()

from sklearn.metrics import roc_auc_score

print("ROC:", roc_auc_score(y_te, y_prob))
print("PR:", average_precision_score(y_te, y_prob))

In [ ]:
model.save("data/lstm_72h.keras")